<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula06d%20-%20mnist%20conv%20torch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow.keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [2]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [7]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

class TorchCNN2D(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=4)
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(in_features=4*25*25, out_features=512)
    self.fc2 = nn.Linear(in_features=512, out_features=num_classes)
  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.flatten(x)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  def __init__(self, epochs=5, batch_size=128, model_fabric=TorchCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = torch.unique(torch.tensor(y), return_inverse=True)
    self.model = self.model_fabric(num_classes=len(self.labels)).to(device)
    self.optimizer = optim.RMSprop(self.model.parameters(), lr=0.001)
    self.loss_fn = nn.CrossEntropyLoss()
    train_dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(ids, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
    for epoch in range(self.epochs):
      for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        self.optimizer.zero_grad()
        y_pred = self.model(batch_X)
        loss = self.loss_fn(y_pred, batch_y)
        loss.backward()
        self.optimizer.step()
    return self

  def predict(self, X):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    y_pred = self.model(X)
    return self.labels[torch.argmax(y_pred, dim=1).cpu().numpy()]


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

class Divide255(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X / 255.0

class Shape2Torch(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X.reshape((-1, 1, 28, 28))

pipeline = Pipeline([
    ("scaler", Divide255()),
    ("shape", Shape2Torch()),
    ("model", TorchWrappedNN(model_fabric=TorchCNN2D))
])

pipeline.fit(train_images, train_labels)
y_pred = pipeline.predict(test_images)
accuracy_score(test_labels, y_pred)

/tmp/ipykernel_3255/1195632135.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(ids, dtype=torch.long))
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


0.9802

In [10]:
class TorchCNN2D(nn.Module):
  def __init__(self, input_shape, num_classes):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=input_shape[0],
                           out_channels=32, kernel_size=3)
    self.pool1 = nn.MaxPool2d(kernel_size=2)
    self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
    self.pool2 = nn.MaxPool2d(kernel_size=2)
    self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(in_features=1152, out_features=num_classes)
  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool1(x)
    x = F.relu(self.conv2(x))
    x = self.pool2(x)
    x = F.relu(self.conv3(x))
    x = self.flatten(x)
    x = self.fc(x)
    x = F.softmax(x, dim=1)
    return x

input_shape = (1, 28, 28)
num_classes = 10
model = TorchCNN2D(input_shape, num_classes)
print(model)

TorchCNN2D(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc): Linear(in_features=1152, out_features=10, bias=True)
)


In [12]:
class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  def __init__(self, epochs=5, batch_size=128, model_fabric=TorchCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = torch.unique(torch.tensor(y), return_inverse=True)
    self.model = self.model_fabric(input_shape=X.shape[1:], num_classes=len(self.labels)).to(device)
    self.optimizer = optim.RMSprop(self.model.parameters(), lr=0.001)
    self.loss_fn = nn.CrossEntropyLoss()
    train_dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(ids, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
    for epoch in range(self.epochs):
      for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        self.optimizer.zero_grad()
        y_pred = self.model(batch_X)
        loss = self.loss_fn(y_pred, batch_y)
        loss.backward()
        self.optimizer.step()
    return self

  def predict(self, X):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    y_pred = self.model(X)
    return self.labels[torch.argmax(y_pred, dim=1).cpu().numpy()]

pipeline = Pipeline([
    ("scaler", Divide255()),
    ("shape", Shape2Torch()),
    ("model", TorchWrappedNN(epochs=10, model_fabric=TorchCNN2D))
])

pipeline.fit(train_images, train_labels)
y_pred = pipeline.predict(test_images)
accuracy_score(test_labels, y_pred)

/tmp/ipykernel_3255/2941036202.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(ids, dtype=torch.long))
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


0.9906